Установка и импорты

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


Загрузка датасета web_questions

In [2]:
ds = tfds.load('web_questions', split=['train', 'test'])
train_data_raw, test_data_raw = ds

print(f"Датасет успешно загружен!")
print(f"Train examples: {len(train_data_raw)}")
print(f"Test examples: {len(test_data_raw)}")

Датасет успешно загружен!
Train examples: 3778
Test examples: 2032


Функция для просмотра примеров

In [3]:
def show_samples(dataset, n=3):
    for i, example in enumerate(dataset.take(n)):
        question = example['question'].numpy().decode() if hasattr(example['question'], 'numpy') else example['question']
        answers = example['answers'].numpy() if hasattr(example['answers'], 'numpy') else example['answers']
        if isinstance(answers, np.ndarray):
            answers = [a.decode() if isinstance(a, bytes) else a for a in answers]
        print(f"Пример {i+1}:")
        print(f"  Вопрос: {question}")
        print(f"  Ответы: {answers}")
        print()

print("\nПримеры из обучающей выборки:")
show_samples(train_data_raw)



Примеры из обучающей выборки:
Пример 1:
  Вопрос: who is the ravens quarterback 2012?
  Ответы: ['Joe Flacco']

Пример 2:
  Вопрос: what political party was henry clay?
  Ответы: ['Whig Party']

Пример 3:
  Вопрос: where to stay in thailand koh samui?
  Ответы: ['UTC+07:00']



Предобработка текста

In [4]:

# Параметры
VOCAB_SIZE = 8000
MAX_LEN_QUESTION = 30
MAX_LEN_ANSWER = 20
EMBEDDING_DIM = 128
NUM_LAYERS = 4
NUM_HEADS = 8
FF_DIM = 512
DROPOUT_RATE = 0.1

# Функция очистки текста
def clean_text(text):
    if isinstance(text, bytes):
        text = text.decode()
    elif not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9?.!,¿']+", " ", text)
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r'[" "]+', " ", text)
    return text.strip()

# Подготовка данных
def prepare_data(dataset):
    questions = []
    answers = []

    # Преобразуем dataset в список для итерации
    for example in tfds.as_numpy(dataset):
        q = example['question']
        a = example['answers'][0]  # берем первый ответ

        # Декодируем байты в строку
        if isinstance(q, bytes):
            q = q.decode('utf-8')
        if isinstance(a, bytes):
            a = a.decode('utf-8')

        questions.append(clean_text(q))
        answers.append(clean_text(a))

    return questions, answers

print("Подготовка данных...")
train_questions, train_answers = prepare_data(train_data_raw)
test_questions, test_answers = prepare_data(test_data_raw)

# Добавляем специальные токены к ответам
train_answers = ["<start> " + a + " <end>" for a in train_answers]
test_answers = ["<start> " + a + " <end>" for a in test_answers]

print(f"Всего обучающих примеров: {len(train_questions)}")
print(f"Всего тестовых примеров: {len(test_questions)}")
print(f"\nПример вопроса: {train_questions[0]}")
print(f"Пример ответа: {train_answers[0]}")

Подготовка данных...
Всего обучающих примеров: 3778
Всего тестовых примеров: 2032

Пример вопроса: who is the ravens quarterback 2012 ?
Пример ответа: <start> joe flacco <end>


Создание токенизатора

In [5]:
# Токенизатор для вопросов
tokenizer_q = tf.keras.preprocessing.text.Tokenizer(
    num_words=VOCAB_SIZE,
    filters='',
    oov_token='<unk>'
)
tokenizer_q.fit_on_texts(train_questions)

# Токенизатор для ответов
tokenizer_a = tf.keras.preprocessing.text.Tokenizer(
    num_words=VOCAB_SIZE,
    filters='',
    oov_token='<unk>'
)
tokenizer_a.fit_on_texts(train_answers)

# Получаем специальные токены
start_token = tokenizer_a.word_index.get('<start>', 1)
end_token = tokenizer_a.word_index.get('<end>', 2)
pad_token = 0

# Преобразуем в последовательности
q_sequences = tokenizer_q.texts_to_sequences(train_questions)
a_sequences = tokenizer_a.texts_to_sequences(train_answers)

q_data = tf.keras.preprocessing.sequence.pad_sequences(
    q_sequences, maxlen=MAX_LEN_QUESTION, padding='post'
)
a_data = tf.keras.preprocessing.sequence.pad_sequences(
    a_sequences, maxlen=MAX_LEN_ANSWER, padding='post'
)

# Для тестовых данных
q_test_sequences = tokenizer_q.texts_to_sequences(test_questions)
a_test_sequences = tokenizer_a.texts_to_sequences(test_answers)

q_test = tf.keras.preprocessing.sequence.pad_sequences(
    q_test_sequences, maxlen=MAX_LEN_QUESTION, padding='post'
)
a_test = tf.keras.preprocessing.sequence.pad_sequences(
    a_test_sequences, maxlen=MAX_LEN_ANSWER, padding='post'
)

vocab_size_q = min(VOCAB_SIZE, len(tokenizer_q.word_index) + 1)
vocab_size_a = min(VOCAB_SIZE, len(tokenizer_a.word_index) + 1)

print(f"Размер словаря вопросов: {vocab_size_q}")
print(f"Размер словаря ответов: {vocab_size_a}")
print(f"Форма данных вопросов: {q_data.shape}")
print(f"Форма данных ответов: {a_data.shape}")

Размер словаря вопросов: 3678
Размер словаря ответов: 3602
Форма данных вопросов: (3778, 30)
Форма данных ответов: (3778, 20)


Создание датасета для обучения

In [6]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((q_data, a_data))
train_dataset = train_dataset.shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((q_test, a_test))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

Позиционное кодирование

In [7]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_len, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(max_len, d_model)

    def get_angles(self, position, i, d_model):
        angles = 1 / tf.pow(10000.0, (2 * (i // 2)) / tf.cast(d_model, tf.float32))
        return position * angles

    def positional_encoding(self, max_len, d_model):
        angle_rads = self.get_angles(
            position=tf.range(max_len, dtype=tf.float32)[:, tf.newaxis],
            i=tf.range(d_model, dtype=tf.float32)[tf.newaxis, :],
            d_model=d_model
        )

        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])

        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = pos_encoding[tf.newaxis, ...]

        return tf.cast(pos_encoding, tf.float32)

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]

Scaled Dot-Product Attention

In [8]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q, k, v: (batch_size, heads, seq_len, d_k)
    mask: (batch_size, 1, 1, seq_len) или (batch_size, 1, seq_len, seq_len)
    """
    matmul_qk = tf.matmul(q, k, transpose_b=True)

    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

    if mask is not None:
        scaled_attention_logits += (mask * -1e9)

    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
    output = tf.matmul(attention_weights, v)

    return output, attention_weights

Multi-Head Attention

In [9]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)

        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        scaled_attention, attention_weights = scaled_dot_product_attention(
            q, k, v, mask
        )

        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))

        output = self.dense(concat_attention)

        return output, attention_weights

Feed-Forward Network

In [10]:
def feed_forward_network(d_model, dff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ])


Encoder Layer

In [11]:

class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = feed_forward_network(d_model, dff)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, x, training=False, mask=None):  # training с default значением
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

Decoder Layer

In [12]:

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(DecoderLayer, self).__init__()

        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = feed_forward_network(d_model, dff)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)
        self.dropout3 = tf.keras.layers.Dropout(rate)

    def call(self, x, enc_output, training=False, look_ahead_mask=None, padding_mask=None):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(x + attn1)

        attn2, attn_weights_block2 = self.mha2(enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(out1 + attn2)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(out2 + ffn_output)

        return out3, attn_weights_block1, attn_weights_block2

Encoder

In [13]:

class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size,
                 maximum_position_encoding, rate=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(maximum_position_encoding, d_model)

        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, rate)
                           for _ in range(num_layers)]

        self.dropout = tf.keras.layers.Dropout(rate)

    def call(self, x, training=False, mask=None):  # training с default значением
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training=training, mask=mask)

        return x

Decoder

In [14]:

class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size,
                 maximum_position_encoding, rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(maximum_position_encoding, d_model)

        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate)
                           for _ in range(num_layers)]

        self.dropout = tf.keras.layers.Dropout(rate)

    def call(self, x, enc_output, training=False, look_ahead_mask=None, padding_mask=None):
        seq_len = tf.shape(x)[1]
        attention_weights = {}

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](x, enc_output,
                                                   training=training,
                                                   look_ahead_mask=look_ahead_mask,
                                                   padding_mask=padding_mask)

            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2

        return x, attention_weights

Полный трансформер

In [15]:

class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 vocab_size_q, vocab_size_a,
                 max_len_q, max_len_a, rate=0.1):
        super(Transformer, self).__init__()

        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                               vocab_size_q, max_len_q, rate)

        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                               vocab_size_a, max_len_a, rate)

        self.final_layer = tf.keras.layers.Dense(vocab_size_a)

    def call(self, inputs, training=False):  # training должен быть именованным
        inp, tar = inputs

        enc_padding_mask = self.create_padding_mask(inp)

        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])
        dec_target_padding_mask = self.create_padding_mask(tar)
        combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

        # ВАЖНО: передаем training как именованный аргумент
        enc_output = self.encoder(inp, training=training, mask=enc_padding_mask)

        dec_output, attention_weights = self.decoder(
            tar, enc_output,
            training=training,  # именованный аргумент
            look_ahead_mask=combined_mask,
            padding_mask=enc_padding_mask
        )

        final_output = self.final_layer(dec_output)

        return final_output

    def create_padding_mask(self, seq):
        seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return seq[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask

Создание модели

In [16]:
transformer = Transformer(
    num_layers=NUM_LAYERS,
    d_model=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    dff=FF_DIM,
    vocab_size_q=vocab_size_q,
    vocab_size_a=vocab_size_a,
    max_len_q=MAX_LEN_QUESTION,
    max_len_a=MAX_LEN_ANSWER,
    rate=DROPOUT_RATE
)

# Строим модель
dummy_input = tf.zeros((1, MAX_LEN_QUESTION), dtype=tf.int32)
dummy_output = tf.zeros((1, MAX_LEN_ANSWER), dtype=tf.int32)
_ = transformer((dummy_input, dummy_output), training=False)

print("Модель создана успешно!")
print(f"Всего параметров: {transformer.count_params():,}")

Модель создана успешно!
Всего параметров: 3,247,890


Функция потерь и оптимизатор

In [17]:

class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # ИСПРАВЛЕНИЕ: преобразуем step в float32
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

learning_rate = CustomSchedule(EMBEDDING_DIM)
optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

Обучение

In [18]:

@tf.function
def train_step(inp, tar):
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    with tf.GradientTape() as tape:
        predictions = transformer((inp, tar_inp), training=True)
        loss = loss_function(tar_real, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    train_loss(loss)
    train_accuracy(tar_real, predictions)

EPOCHS = 60

print("Начинаем обучение...")
for epoch in range(EPOCHS):
    start = time.time()

    # ИСПРАВЛЕНО: reset_states() -> reset_state()
    train_loss.reset_state()
    train_accuracy.reset_state()

    for batch, (inp, tar) in enumerate(train_dataset):
        train_step(inp, tar)

        if batch % 10 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')

    print(f'Epoch {epoch+1} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')
    print(f'Время эпохи: {time.time() - start:.2f} сек')

print("Обучение завершено!")

Начинаем обучение...
Epoch 1 Batch 0 Loss 8.0697 Accuracy 0.0066
Epoch 1 Batch 10 Loss 8.0517 Accuracy 0.0025
Epoch 1 Batch 20 Loss 8.0237 Accuracy 0.0063
Epoch 1 Batch 30 Loss 7.9813 Accuracy 0.0185
Epoch 1 Batch 40 Loss 7.9389 Accuracy 0.0268
Epoch 1 Batch 50 Loss 7.8937 Accuracy 0.0319
Epoch 1 Batch 60 Loss 7.8472 Accuracy 0.0353
Epoch 1 Batch 70 Loss 7.8022 Accuracy 0.0377
Epoch 1 Batch 80 Loss 7.7569 Accuracy 0.0396
Epoch 1 Batch 90 Loss 7.7137 Accuracy 0.0410
Epoch 1 Batch 100 Loss 7.6793 Accuracy 0.0421
Epoch 1 Batch 110 Loss 7.6408 Accuracy 0.0431
Epoch 1 Loss 7.6137 Accuracy 0.0437
Время эпохи: 38.81 сек
Epoch 2 Batch 0 Loss 7.3236 Accuracy 0.0526
Epoch 2 Batch 10 Loss 7.2389 Accuracy 0.0526
Epoch 2 Batch 20 Loss 7.2021 Accuracy 0.0526
Epoch 2 Batch 30 Loss 7.1714 Accuracy 0.0526
Epoch 2 Batch 40 Loss 7.1556 Accuracy 0.0526
Epoch 2 Batch 50 Loss 7.1296 Accuracy 0.0526
Epoch 2 Batch 60 Loss 7.1063 Accuracy 0.0526
Epoch 2 Batch 70 Loss 7.0835 Accuracy 0.0526
Epoch 2 Batch 80 Los

Функция для генерации ответов

In [29]:
def generate_answer_no_repeat(question, max_length=5):
    question_clean = clean_text(question)
    seq = tokenizer_q.texts_to_sequences([question_clean])
    padded = tf.keras.preprocessing.sequence.pad_sequences(
        seq, maxlen=MAX_LEN_QUESTION, padding='post'
    )

    decoder_input = tf.expand_dims([start_token], 0)
    result = []
    last_token = None

    for i in range(max_length):
        predictions = transformer((padded, decoder_input), training=False)
        next_token_logits = predictions[0, -1, :]

        # Запрещаем повторять последний токен
        if last_token is not None:
            next_token_logits = tf.tensor_scatter_nd_add(
                next_token_logits,
                [[last_token]],
                [-100.0]
            )

        predicted_id = tf.argmax(next_token_logits).numpy()

        if predicted_id == end_token:
            break

        result.append(tokenizer_a.index_word[predicted_id])
        decoder_input = tf.concat([decoder_input, [[predicted_id]]], axis=-1)
        last_token = predicted_id

    return ' '.join(result)

Финальное тестирование

In [31]:
test_questions = [
    "who is the ravens quarterback 2012 ?",
    "what political party was henry clay ?",
    "where to stay in thailand koh samui ?",
    "what countries are included in the continent of asia ?",
    "what movies does tupac act in ?"
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer_no_repeat(q)}")
    print()

Q: who is the ravens quarterback 2012 ?
A: joe flacco

Q: what political party was henry clay ?
A: whig party

Q: where to stay in thailand koh samui ?
A: utc 07 00

Q: what countries are included in the continent of asia ?
A: bahrain

Q: what movies does tupac act in ?
A: murder was the case

